In [1]:
# Imports — we only need arxiv and pandas to explore the API
import arxiv
import pandas as pd

In [17]:
# What does the arxiv module expose? (public names only)
public = [x for x in dir(arxiv) if not x.startswith("_")]
print("Public names in arxiv:", public)

Public names in arxiv: ['ArxivError', 'Client', 'Enum', 'Generator', 'HTTPError', 'Iterator', 'Result', 'Search', 'SortCriterion', 'SortOrder', 'TYPE_CHECKING', 'UnexpectedEmptyPageError', 'annotations', 'datetime', 'feedparser', 'itertools', 'logger', 'logging', 'math', 'os', 're', 'requests', 'time', 'timedelta', 'timegm', 'timezone', 'urlencode', 'urlparse', 'urlretrieve', 'warnings']


In [18]:
# Sort options: how can we order search results?
print("SortCriterion:", list(arxiv.SortCriterion))
print("SortOrder:", list(arxiv.SortOrder))

SortCriterion: [<SortCriterion.Relevance: 'relevance'>, <SortCriterion.LastUpdatedDate: 'lastUpdatedDate'>, <SortCriterion.SubmittedDate: 'submittedDate'>]
SortOrder: [<SortOrder.Ascending: 'ascending'>, <SortOrder.Descending: 'descending'>]


In [19]:
# Run a minimal search to get one result so we can inspect what one "paper" object has
search = arxiv.Search(
    query="cat:cs.AI",
    max_results=1,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)
client = arxiv.Client(page_size=10)
one_result = next(client.results(search))

In [20]:
# What attributes does one result (paper) have? These are the fields we can extract.
print("Result type:", type(one_result))
print("\nAttributes (public):", [x for x in dir(one_result) if not x.startswith("_")])

Result type: <class 'arxiv.Result'>

Attributes (public): ['Author', 'Link', 'MissingFieldError', 'authors', 'categories', 'comment', 'doi', 'download_pdf', 'download_source', 'entry_id', 'get_short_id', 'journal_ref', 'links', 'pdf_url', 'primary_category', 'published', 'source_url', 'summary', 'title', 'updated']


In [21]:
# Print each field we care about for one paper (so we know how to extract them)
print("entry_id:", one_result.entry_id)
print("title:", one_result.title[:80] + "..." if len(one_result.title) > 80 else one_result.title)
print("summary (abstract) length:", len(one_result.summary))
print("published:", one_result.published)
print("updated:", one_result.updated)
print("primary_category:", one_result.primary_category)
print("categories (all):", one_result.categories)
print("authors:", [a.name for a in one_result.authors[:3]], "...")
print("doi:", one_result.doi)
print("journal_ref:", one_result.journal_ref)
print("pdf_url:", one_result.pdf_url)

entry_id: http://arxiv.org/abs/2602.20159v1
title: A Very Big Video Reasoning Suite
summary (abstract) length: 1395
published: 2026-02-23 18:59:41+00:00
updated: 2026-02-23 18:59:41+00:00
primary_category: cs.CV
categories (all): ['cs.CV', 'cs.AI', 'cs.LG', 'cs.MM', 'cs.RO']
authors: ['Maijunxian Wang', 'Ruisi Wang', 'Juyi Lin'] ...
doi: None
journal_ref: None
pdf_url: https://arxiv.org/pdf/2602.20159v1


In [22]:
# Tech categories used in this project (from backend)
from backend.core.constants import TECH_CATEGORIES

print("Number of tech categories we use:", len(TECH_CATEGORIES))
print("\nCategory code -> human-readable name:")
for code, name in list(TECH_CATEGORIES.items())[:10]:
    print(f"  {code} -> {name}")
print("  ...")
print("\nAll codes:", list(TECH_CATEGORIES.keys()))

Number of tech categories we use: 22

Category code -> human-readable name:
  cs.AI -> Artificial Intelligence
  cs.LG -> Machine Learning
  cs.NE -> Neural Networks & Evolutionary
  cs.CV -> Computer Vision
  cs.GR -> Graphics
  cs.CL -> Natural Language Processing
  cs.IR -> Information Retrieval
  cs.DB -> Databases
  cs.DS -> Data Structures & Algorithms
  cs.SE -> Software Engineering
  ...

All codes: ['cs.AI', 'cs.LG', 'cs.NE', 'cs.CV', 'cs.GR', 'cs.CL', 'cs.IR', 'cs.DB', 'cs.DS', 'cs.SE', 'cs.PL', 'cs.OS', 'cs.DC', 'cs.AR', 'cs.CR', 'cs.NI', 'cs.HC', 'cs.CY', 'cs.RO', 'cs.SY', 'cs.CC', 'cs.GT']


In [23]:
# Query by year: arXiv uses submittedDate in format [YYYYMMDD TO YYYYMMDD]
year = 2024
category = "cs.AI"
start_date = f"{year}0101"
end_date = f"{year}1231"
query = f"cat:{category} AND submittedDate:[{start_date} TO {end_date}]"
print("Example query:", query)

search_2024 = arxiv.Search(
    query=query,
    max_results=10,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)
client_2024 = arxiv.Client(page_size=10)
sample_results = list(client_2024.results(search_2024))
print("Number of results (max_results=10):", len(sample_results))
print("So we can filter by year and category; arXiv has papers from early 1990s to present.")

Example query: cat:cs.AI AND submittedDate:[20240101 TO 20241231]
Number of results (max_results=10): 10
So we can filter by year and category; arXiv has papers from early 1990s to present.


In [24]:
# Manually build one dict per result — no function, just a loop
rows = []
for r in sample_results:
    rows.append(
        {
            "arxiv_id": r.entry_id.split("/")[-1],
            "title": r.title.replace("\n", " ").strip(),
            "abstract": r.summary.replace("\n", " ").strip(),
            "published_date": r.published.strftime("%Y-%m-%d"),
            "primary_category": r.primary_category,
            "all_categories": ", ".join(r.categories),
            "authors": ", ".join(a.name for a in r.authors[:10]),
        }
    )

# Show first row (so we see the extracted fields) and then the DataFrame
print("First row (extracted fields):", rows[0] if rows else "no rows")
df_preview = pd.DataFrame(rows)
df_preview

First row (extracted fields): {'arxiv_id': '2501.00669v1', 'title': 'Leaf diseases detection using deep learning methods', 'abstract': 'This study, our main topic is to devlop a new deep-learning approachs for plant leaf disease identification and detection using leaf image datasets. We also discussed the challenges facing current methods of leaf disease detection and how deep learning may be used to overcome these challenges and enhance the accuracy of disease detection. Therefore, we have proposed a novel method for the detection of various leaf diseases in crops, along with the identification and description of an efficient network architecture that encompasses hyperparameters and optimization methods. The effectiveness of different architectures was compared and evaluated to see the best architecture configuration and to create an effective model that can quickly detect leaf disease. In addition to the work done on pre-trained models, we proposed a new model based on CNN, which pro

,arxiv_id,title,abstract,published_date,primary_category,all_categories,authors
0,2501.00669v1,Leaf diseases detection using deep learning me...,"This study, our main topic is to devlop a new ...",2024-12-31,cs.LG,"cs.LG, cs.AI, cs.CV",El Houcine El Fatimi
1,2501.00664v3,Grade Inflation in Generative Models,"Generative models hold great potential, but on...",2024-12-31,cs.AI,"cs.AI, cs.LG, stat.ML","Phuc Nguyen, Miao Li, Alexandra Morgan, Rima A..."
2,2501.00663v1,Titans: Learning to Memorize at Test Time,Over more than a decade there has been an exte...,2024-12-31,cs.LG,"cs.LG, cs.AI, cs.CL","Ali Behrouz, Peilin Zhong, Vahab Mirrokni"
3,2501.00644v1,Efficient Standardization of Clinical Notes us...,Clinician notes are a rich source of patient i...,2024-12-31,cs.CL,"cs.CL, cs.AI","Daniel B. Hier, Michael D. Carrithers, Thanh S..."
4,2501.00642v1,Enabling New HDLs with Agents,Large Language Models (LLMs) based agents are ...,2024-12-31,cs.AR,"cs.AR, cs.AI, cs.LG, cs.PL","Mark Zakharov, Farzaneh Rabiei Kashanaki, Jose..."
5,2501.01994v1,Fuzzy Model Identification and Self Learning w...,This paper develops a smooth model identificat...,2024-12-31,eess.SY,"eess.SY, cs.AI","Ebrahim Navid Sadjadi, Jesus Garcia, Jose M. M..."
6,2501.05464v2,LLM-MedQA: Enhancing Medical Question Answerin...,Accurate and efficient question-answering syst...,2024-12-31,cs.CL,"cs.CL, cs.AI, cs.IR","Hang Yang, Hao Chen, Hui Guo, Yineng Chen, Chi..."
7,2501.00619v1,A Study on Context Length and Efficient Transf...,Biomedical imaging modalities often produce hi...,2024-12-31,cs.CV,"cs.CV, cs.AI, cs.LG","Sarah M. Hooper, Hui Xue"
8,2501.00601v2,DreamDrive: Generative 4D Scene Modeling from ...,Synthesizing photo-realistic visual observatio...,2024-12-31,cs.CV,"cs.CV, cs.AI, cs.GR","Jiageng Mao, Boyi Li, Boris Ivanovic, Yuxiao C..."
9,2501.00599v3,VideoRefer Suite: Advancing Spatial-Temporal O...,Video Large Language Models (Video LLMs) have ...,2024-12-31,cs.CV,"cs.CV, cs.AI, cs.LG","Yuqian Yuan, Hang Zhang, Wentong Li, Zesen Che..."
